In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/1056lab-car-price-prediction-2026/train.csv', index_col=0, encoding='latin1')
train_df

In [ ]:
test_df = pd.read_csv('/kaggle/input/competitions/1056lab-car-price-prediction-2026/test.csv', index_col=0, encoding='latin1')
test_df

In [ ]:
train_df.dtypes

In [ ]:
import matplotlib.pyplot as plt

x1 = train_df['Production_year'].to_numpy()
x2 = test_df['Production_year'].to_numpy()

plt.style.use('ggplot')
plt.figure()
plt.hist(x1, alpha=0.8, log=True, label='train')
plt.hist(x2, alpha=0.8, log=True, label='test')
plt.legend()
plt.title('Production_year')
plt.xlabel('Production_year')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x1 = train_df['Mileage_km'].to_numpy()
x2 = test_df['Mileage_km'].to_numpy()

plt.style.use('ggplot')
plt.figure()
plt.hist(x1, alpha=0.8, log=True, label='train')
plt.hist(x2, alpha=0.8, log=True, label='test')
plt.legend()
plt.title('Mileage_km')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

x1 = train_df['Power_HP'].to_numpy()
x2 = test_df['Power_HP'].to_numpy()

plt.style.use('ggplot')
plt.figure()
plt.hist(x1, alpha=0.8, log=True, label='train')
plt.hist(x2, alpha=0.8, log=True, label='test')
plt.legend()
plt.title('Power_HP')
plt.show()

In [ ]:

train_df

In [ ]:
train_df.isna().sum()

In [ ]:
test_df.isna().sum()

In [ ]:
y = train_df['Price'].to_numpy()
y = np.log1p(y)
train_df['Price'] = y
# --- 特徴量追加 ---
train_df['car_age'] = 2026 - train_df['Production_year']
test_df['car_age'] = 2026 - test_df['Production_year']

train_df['km_per_year'] = train_df['Mileage_km'] / (train_df['car_age'] + 1)
test_df['km_per_year'] = test_df['Mileage_km'] / (test_df['car_age'] + 1)

train_df['power_per_l'] = train_df['Power_HP'] / train_df['Displacement_cm3']
test_df['power_per_l'] = test_df['Power_HP'] / test_df['Displacement_cm3']
train_df

In [ ]:
X_train = train_df.drop('Price', axis=1)
y_train = train_df['Price']

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


numeric_features = [
    'Mileage_km',
    'Power_HP',
    'Production_year',
    'Displacement_cm3',
    'car_age',
    'km_per_year',
    'power_per_l'
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


categorical_features = [
    'Transmission',
    'Fuel_type',
    'Condition',
    'Vehicle_brand',
    'Vehicle_model',
    'Vehicle_generation'

]

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

train_df[categorical_features] = train_df[categorical_features].astype(str)
test_df[categorical_features] = test_df[categorical_features].astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import cross_val_score, KFold
from lightgbm import LGBMRegressor

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("lgbm", LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=-1,
            random_state=42
        ))
    ]
)
cv = KFold(n_splits=10, random_state=42, shuffle=True)

scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
score = -scores.mean()

print(scores)
print(score)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import cross_val_score, KFold
import optuna

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
    }

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("lgbm", LGBMRegressor(**params, random_state=42, n_jobs=-1))
        ]
    )

    cv = KFold(n_splits=3, shuffle=True, random_state=42)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error"
    )

    return -scores.mean()
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100)

In [ ]:
print(study.best_value)
print(study.best_params)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("lgbm", LGBMRegressor(**study.best_params, random_state=42))
    ]
)

model.fit(X_train, y_train)


In [ ]:

X_test = test_df[X_train.columns]

p_test = model.predict(X_test)
p_test

In [ ]:
p_test = np.expm1(p_test)
p_test

In [ ]:
submit_df = pd.read_csv('/kaggle/input/competitions/1056lab-car-price-prediction-2026/sample_submission.csv', index_col=0)
submit_df

In [ ]:
submit_df['Price'] = p_test
submit_df

In [ ]:
submit_df.to_csv('submission.csv')